## MODELOS ENTRENADOS

In [5]:
from scipy.ndimage import binary_erosion
from scipy.spatial.distance import directed_hausdorff
import numpy as np
import cv2

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

def resize_for_hausdorff(pred, gt, max_size=512):
    if max(pred.shape) > max_size:
        scale = max_size / max(pred.shape)
        new_h = int(pred.shape[0] * scale)
        new_w = int(pred.shape[1] * scale)
        pred = cv2.resize(pred.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        gt   = cv2.resize(gt.astype(np.uint8),   (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return pred, gt

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [6]:
import time
import torch

def measure_inference_mapillary(predictor, central_point, labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    masks, scores, _ = predictor.predict(point_coords=central_point, point_labels=labels)
    latency = (time.time() - start) * 1000  # Está en ms

    vram = 0
    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before

    return masks, scores, latency, vram


In [7]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [8]:
import os
import shutil
import random

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"

def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "images")
    masks_dir = os.path.join(dataset, "instances")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "masks",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "images", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".png"), os.path.join(dataset, "masks",  split, name + ".png"))

split_dataset(dataset)


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry
import cv2
import numpy as np
import os
import time
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)

        cache_path = os.path.join(dataset_path, f"cache_{split}.json")

        if os.path.exists(cache_path):
            with open(cache_path) as f:
                self.samples = json.load(f)
            return

        self.samples = []
        for img_file in os.listdir(self.images_dir):
            if not img_file.endswith(".jpg"):
                continue
            img_name  = os.path.splitext(img_file)[0]
            img_path  = os.path.join(self.images_dir, img_file)
            mask_path = os.path.join(self.masks_dir, img_name + ".png")
            if not os.path.exists(mask_path):
                continue

            inst_img = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
            inst_img = np.squeeze(inst_img)
            if inst_img is None:
                continue

            instance_ids = np.unique(inst_img)
            instance_ids = instance_ids[instance_ids != 0]

            if len(instance_ids) > 5:
                rng = np.random.default_rng(42)
                instance_ids = rng.choice(instance_ids, size=5, replace=False)

            for inst_id in instance_ids:
                coords = np.argwhere(inst_img == inst_id)
                if len(coords) < 100:
                    continue
                self.samples.append((img_path, mask_path, int(inst_id)))

        with open(cache_path, "w") as f:
            json.dump(self.samples, f)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, inst_id = self.samples[idx]

        inst_img = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        inst_img = np.squeeze(inst_img)
        gt_bool  = (inst_img == inst_id)

        coords         = np.argwhere(gt_bool)
        ymin, xmin     = coords.min(axis=0)
        ymax, xmax     = coords.max(axis=0)
        orig_h, orig_w = inst_img.shape[:2]

        cx = xmin + (xmax - xmin) / 2
        cy = ymin + (ymax - ymin) / 2

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask_resized = cv2.resize(gt_bool.astype(np.uint8), (256, 256), interpolation=cv2.INTER_NEAREST)
        mask = torch.tensor(mask_resized).float().unsqueeze(0)

        cx_scaled = cx * (self.img_size / orig_w)
        cy_scaled = cy * (self.img_size / orig_h)

        point = torch.tensor([[cx_scaled, cy_scaled]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


def train_sam(dataset_path, weights_path, output_weights, vit, epochs=50, batch_size=4):
    sam = sam_model_registry[vit](checkpoint=weights_path)
    sam.to(device)

    sam.image_encoder.eval()
    sam.prompt_encoder.eval()
    sam.mask_decoder.train()

    for param in sam.image_encoder.parameters():
        param.requires_grad = False
    for param in sam.prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam.mask_decoder.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    start = time.time()
    for images, masks, points, labels in train_loader:
        pass
    print(f"Solo carga: {(time.time()-start)/60:.2f} min")

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            images, masks  = images.to(device), masks.to(device)
            points, labels = points.to(device), labels.to(device)

            with torch.no_grad():
                image_embeddings = sam.image_encoder(images)

            loss_batch = 0
            for i in range(images.shape[0]):
                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam.prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                low_res_masks, _ = sam.mask_decoder(
                    image_embeddings=image_embeddings[i].unsqueeze(0),
                    image_pe=sam.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save(sam.mask_decoder.state_dict(), output_weights)
    return output_weights


dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"

print("Construyendo caché...")
SegDataset(dataset, "train")
SegDataset(dataset, "val")
print("Caché listo")

Construyendo caché...
Caché listo


## SAM 1 BASE TRAS ENTRENAMIENTO

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import torch
import time

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "D:\\pesos_fine_tuned\\sam_b_mapillary.pt"


def evaluate_model(dataset, weights_path, base_weights_path):
    sam = sam_model_registry["vit_b"](checkpoint=base_weights_path)
    decoder_state = torch.load(weights_path, map_location=device)
    missing, unexpected = sam.mask_decoder.load_state_dict(decoder_state, strict=True)
    if missing or unexpected:
        raise RuntimeError(f"Error cargando pesos del decoder.\nFaltantes: {missing}\nInesperados: {unexpected}")
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "images", "test")
    masks_dir  = os.path.join(dataset, "masks",  "test")

    for img_file in os.listdir(images_dir):
        img_name  = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(mask_path):
            continue

        inst_img = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        inst_img = np.squeeze(inst_img)
        if inst_img is None:
            continue

        instance_ids = np.unique(inst_img)
        instance_ids = instance_ids[instance_ids != 0]

        if len(instance_ids) > 5:
            rng = np.random.default_rng(42)
            instance_ids = rng.choice(instance_ids, size = 5, replace = False)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        predictor.set_image(image)

        for inst_id in instance_ids:
            gt_mask = (inst_img == inst_id)
            coords = np.argwhere(gt_mask)
            if len(coords) < 100:
                continue
        
            ymin, xmin = coords.min(axis=0)
            ymax, xmax = coords.max(axis=0) 
            central_point = np.array([[xmin + (xmax - xmin) / 2, ymin + (ymax - ymin) / 2]])

            masks, scores, latency, vram = measure_inference_mapillary(predictor, central_point, np.array([1]))

            if masks is None or len(masks) == 0:
                continue

            best_idx  = np.argmax(scores)
            pred_mask = masks[best_idx].astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            f1_scores.append(f1)
            recall_scores.append(recall)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_b_mapillary"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results
         

start_train = time.time()
trained_weights = train_sam(dataset, weights, output_weights, vit="vit_b")
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights, weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Usando GPU: NVIDIA GeForce RTX 3090


c:\users\danieltalavera\desktop\trabajo_fin_de_grado\segment-anything\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.loa

Solo carga: 16.23 min
Epoch 1/50 - Loss: 0.0657
Epoch 2/50 - Loss: 0.0562
Epoch 3/50 - Loss: 0.0518
Epoch 4/50 - Loss: 0.0534
Epoch 5/50 - Loss: 0.0488
Epoch 6/50 - Loss: 0.0494
Epoch 7/50 - Loss: 0.0477
Epoch 8/50 - Loss: 0.0455
Epoch 9/50 - Loss: 0.0455
Epoch 10/50 - Loss: 0.0443
Epoch 11/50 - Loss: 0.0445
Epoch 12/50 - Loss: 0.0439
Epoch 13/50 - Loss: 0.0413
Epoch 14/50 - Loss: 0.0407
Epoch 15/50 - Loss: 0.0404
Epoch 16/50 - Loss: 0.0408
Epoch 17/50 - Loss: 0.0381
Epoch 18/50 - Loss: 0.0374
Epoch 19/50 - Loss: 0.0370
Epoch 20/50 - Loss: 0.0347
Epoch 21/50 - Loss: 0.0306
Epoch 22/50 - Loss: 0.0301
Epoch 23/50 - Loss: 0.0283
Epoch 24/50 - Loss: 0.0266
Epoch 25/50 - Loss: 0.0263
Epoch 26/50 - Loss: 0.0266
Epoch 27/50 - Loss: 0.0255
Epoch 28/50 - Loss: 0.0253
Epoch 29/50 - Loss: 0.0251
Epoch 30/50 - Loss: 0.0235
Epoch 31/50 - Loss: 0.0246
Epoch 32/50 - Loss: 0.0234
Epoch 33/50 - Loss: 0.0229
Epoch 34/50 - Loss: 0.0223
Epoch 35/50 - Loss: 0.0221
Epoch 36/50 - Loss: 0.0224
Epoch 37/50 - L

## SAM 1 GRANDE

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import torch
import time

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Mapillary_Vistas\\validation"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "D:\\pesos_fine_tuned\\sam_l_mapillary.pt"


def evaluate_model(dataset, weights_path, base_weights_path):
    sam = sam_model_registry["vit_l"](checkpoint=base_weights_path)
    decoder_state = torch.load(weights_path, map_location=device)
    missing, unexpected = sam.mask_decoder.load_state_dict(decoder_state, strict=True)
    if missing or unexpected:
        raise RuntimeError(f"Error cargando pesos del decoder.\nFaltantes: {missing}\nInesperados: {unexpected}")
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "images", "test")
    masks_dir  = os.path.join(dataset, "masks",  "test")

    for img_file in os.listdir(images_dir):
        img_name  = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(mask_path):
            continue

        inst_img = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        inst_img = np.squeeze(inst_img)
        if inst_img is None:
            continue

        instance_ids = np.unique(inst_img)
        instance_ids = instance_ids[instance_ids != 0]

        if len(instance_ids) > 5:
            rng = np.random.default_rng(42)
            instance_ids = rng.choice(instance_ids, size = 5, replace = False)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        predictor.set_image(image)

        for inst_id in instance_ids:
            gt_mask = (inst_img == inst_id)
            coords = np.argwhere(gt_mask)
            if len(coords) < 100:
                continue
        
            ymin, xmin = coords.min(axis=0)
            ymax, xmax = coords.max(axis=0) 
            central_point = np.array([[xmin + (xmax - xmin) / 2, ymin + (ymax - ymin) / 2]])

            masks, scores, latency, vram = measure_inference_mapillary(predictor, central_point, np.array([1]))

            if masks is None or len(masks) == 0:
                continue

            best_idx  = np.argmax(scores)
            pred_mask = masks[best_idx].astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            f1_scores.append(f1)
            recall_scores.append(recall)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_l_mapillary"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results
         

start_train = time.time()
trained_weights = train_sam(dataset, weights, output_weights, vit="vit_l")
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights, weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Usando GPU: NVIDIA GeForce RTX 3090
Epoch 1/50 - Loss: 0.3717
Epoch 2/50 - Loss: 0.3520
Epoch 3/50 - Loss: 0.3362
Epoch 4/50 - Loss: 0.3295
Epoch 5/50 - Loss: 0.3164
Epoch 6/50 - Loss: 0.3082
Epoch 7/50 - Loss: 0.3033
Epoch 8/50 - Loss: 0.2977
Epoch 9/50 - Loss: 0.2958
Epoch 10/50 - Loss: 0.2943
Epoch 11/50 - Loss: 0.2817
Epoch 12/50 - Loss: 0.2754
Epoch 13/50 - Loss: 0.2724
Epoch 14/50 - Loss: 0.2731
Epoch 15/50 - Loss: 0.2652
Epoch 16/50 - Loss: 0.2639


KeyboardInterrupt: 